# Objective

#### Broad level:
This notebook will allow you to take either input .pdb or .cif files and transform them with mini3di into the 3Di language used by Foldseek for downstream use.

#### Specific:
Alphafold predicts 5 models for any 1 given input sequence with no significant difference (to my knowledge) between the structure of each model. This notebook will take all Alphafold-predicted .pdb or .cif protein structural files in a given directory and process the first file for each protein into the 3Di language used by Foldseek. It does so by making use of the `mini3Di` encoder (<https://github.com/althonos/mini3di>). It will then format it into FASTA format and output one 3Di-transformed protein sequence per input model(s). It will take all proteins/orthologs that match a specific protein name and group them as one saved .fas file. (e.g. `fold_species1_proteinA_model_0.cif` and `fold_species2_proteinA_model_5.cif` will be transformed into 3Di language and saved in FASTA format as `proteinA.cif`). Ultimately, this is meant to be used for a modified FAMSA provided by Mifsud et al.

### Notes:
1) Make sure that all orthologs are saved in different folders. (e.g. one directory should be entirely "proteinA" predicted structures and another directory should be entirely "proteinB" predicted structures).
2) This can work for multimers as well and how to do so is detailed below but this is primarily designed for monomers (and I'm not sure why this would be used for multimers but it could be due to my own ignorance).
3) I have not validated this for PDB-resolved structures.

Updated 2025-03-18

## Citations:
1) Foldseek: van Kempen, M., Kim, S.S., Tumescheit, C. et al. Fast and accurate protein structure search with Foldseek. 
    Nat Biotechnol 42, 243–246 (2024). https://doi.org/10.1038/s41587-023-01773-0. Available here: https://github.com/steineggerlab/foldseek.
2) Modified FAMSA: Mifsud, J.C.O., Lytras, S., Oliver, M.R. et al. Mapping glycoprotein structure reveals Flaviviridae evolutionary history. 
    Nature 633, 695–703 (2024). https://doi.org/10.1038/s41586-024-07899-8. Code available here: https://zenodo.org/records/11092288.
3) mini3di: https://github.com/althonos/mini3di

In [ ]:
import os
import mini3di
from collections import OrderedDict
from Bio.PDB import MMCIFParser, PDBParser

def process_alphafold_structures(input_directory, output_filepath, process_all_chains=False):
    """
    Processes Alphafold predicted structure files in input_directory and performs 3Di transformation.
    Makes use of mini3di from https://github.com/althonos/mini3di (original 3Di from Foldseek, cited previously).
    Outputs 3Di-transformed structures in FASTA format for use with modified FAMSA alignment (Mifsud et al.)
    
    Files are expected to follow the naming pattern:
      fold_[token]_model_[integer].cif
      
    The function extracts the [token] (i.e. the part between "fold_" and "_model_")
    from each filename and uses it as the FASTA header.
    
    By default (process_all_chains=False), only the first chain of each structure is processed.
    This produces a header like:
        >[token]
    which is suitable for single-chain conversion.
    
    If you wish to process every chain (process_all_chains=True), then for each chain the
    header will be appended with the chain identifier (e.g., ">[token]_chain_A").
    
    In either case, if multiple files have the same token, only the first encountered file is used.
    This is because Alphafold predicts five structures for one input sequence. 
    So far, to my knowledge, there has been no significant difference shown between the five different predicted models.
    
    Usage examples:
      # For single-chain processing (default):
      process_alphafold_structures("path_to_input_directory", "output_directory/my_sequences.fas")
      
      # For multi-chain processing, uncomment the following line:
      # process_alphafold_structures("path_to_input_directory", "output_directory/my_sequences.fas", process_all_chains=True)
    """
    
    # Initialize parsers and mini3di encoder.
    cif_parser = MMCIFParser(QUIET=True)
    pdb_parser = PDBParser(QUIET=True)
    encoder = mini3di.Encoder()
    
    # Use an OrderedDict to store unique tokens and their FASTA entries (as a list).
    fasta_entries = OrderedDict()
    
    # Walk through the input directory.
    for root, _, files in os.walk(input_directory):
        for filename in sorted(files):
            # Only process files that follow the naming pattern.
            if filename.startswith("fold_") and "_model_" in filename and filename.endswith(".cif"):
                filepath = os.path.join(root, filename)
                # Extract the token between "fold_" and the last occurrence of "_model_"
                token = filename[len("fold_"):filename.rfind("_model_")]
                
                # Skip if we've already processed a file with the same token.
                if token in fasta_entries:
                    continue
                
                try:
                    # Parse the structure using the MMCIFParser.
                    structure = cif_parser.get_structure(filename, filepath)
                    
                    # Initialize the list of FASTA entries for this token.
                    fasta_entries[token] = []
                    
                    if process_all_chains:
                        # Multi-chain processing: process every chain.
                        for chain in structure.get_chains():
                            states = encoder.encode_chain(chain)
                            sequence = encoder.build_sequence(states)
                            header = f">{token}_chain_{chain.get_id()}"
                            fasta_entry = f"{header}\n{sequence}"
                            fasta_entries[token].append(fasta_entry)
                            print(f"Processed: {filename} as token: {token}, chain: {chain.get_id()}")
                    else:
                        # Single-chain processing: process only the first chain.
                        for chain in structure.get_chains():
                            states = encoder.encode_chain(chain)
                            sequence = encoder.build_sequence(states)
                            header = f">{token}"
                            fasta_entry = f"{header}\n{sequence}"
                            fasta_entries[token].append(fasta_entry)
                            print(f"Processed: {filename} as token: {token} (first chain only)")
                            break  # Exit after processing the first chain.
                except Exception as e:
                    print(f"Error processing {filename}: {e}")
    
    # Ensure the directory for the output file exists.
    output_dir = os.path.dirname(output_filepath)
    if output_dir:
        os.makedirs(output_dir, exist_ok=True)
    
    # Write all FASTA entries to the output file.
    with open(output_filepath, "w") as f:
        for entries in fasta_entries.values():
            for entry in entries:
                f.write(entry + "\n")
    
    print(f"Saved FASTA file: {output_filepath}")

# Example usage:
# For single-chain processing (default), which produces headers like ">[token]":
# process_alphafold_structures("path_to_input_directory", "path_to_output_directory/my_wonderful_sequences.fas")

# For multi-chain processing (if desired), which produces headers like ">[token]_chain_A":
# process_alphafold_structures("path_to_input_directory", "path_to_output_directory/my_beautiful_sequences.fas", process_all_chains=True)
